# 04 - Spark basics - Solution


In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("lesson04_spark_basics").getOrCreate()
orders_df = spark.read.option("header", True).option("inferSchema", True).csv("/workspace/dataset/orders.csv")
products_df = spark.read.json("/workspace/dataset/products.json")
orders_df.printSchema()
products_df.printSchema()


In [ ]:
sales_by_category = (
    orders_df.join(products_df, on="product_id", how="left")
    .groupBy("category")
    .agg(
        F.count("*").alias("rows_cnt"),
        F.round(F.sum("amount"), 2).alias("revenue"),
        F.round(F.avg("price"), 2).alias("avg_price"),
    )
    .orderBy(F.col("revenue").desc())
)
sales_by_category.show(20, truncate=False)


In [ ]:
output_path = "file:///workspace/output/lesson04_sales_by_category"
sales_by_category.write.mode("overwrite").parquet(output_path)
spark.read.parquet(output_path).show(10, truncate=False)
